# Data Processing

In [ ]:
import polars as pl

import nwec.utils.excel
from nwec.constants import CLEAN_UTILITY_DATA, MONTHS, RAW_UTILITY_DATA
from nwec.utils.cleaning import clean_utility_data, validate_data

In [ ]:
spreadsheet = RAW_UTILITY_DATA / "IOU 200281 Data.xlsx"
sheet_name = "No. Customers w Arrears"
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, sheet_name)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)

# Get rid of the first row and promote the second to be the DataFrame header
headers = df.slice(1, 1).row(0)
df = df.slice(2).rename(dict(zip(df.columns, [str(col) for col in headers], strict=True)))

In [ ]:
index_cols = [col for col in df.columns if col not in MONTHS]
# Unpivot the month columns while keeping Year as a regular column
df = df.unpivot(on=MONTHS, index=index_cols, variable_name="Month", value_name="Arrearage Count")
# Convert month names to integers (1-12)
df = df.with_columns(pl.col("Month").str.to_datetime("%B").dt.month().alias("Month"))

In [ ]:
# Filter for residential customers and clean up the "Arrearage Count" column
df = df.filter(pl.col("Customer Class").str.contains(r"(?i)res")).drop("Customer Class")
df = clean_utility_data(df, value_column_name="Arrearage Count")
validate_data(df, value_column_name="Arrearage Count")

ValueError: Data validation failed:
  - Zip Code: Found invalid zip codes: ['(blanks)', '(blank)']


In [ ]:
CLEAN_UTILITY_DATA.mkdir(parents=True, exist_ok=True)
processed_path = CLEAN_UTILITY_DATA / "arrearage_counts.arrow"

if processed_path.exists():
    combined_arrearage_counts = pl.read_ipc(processed_path)
    combined_arrearage_counts = pl.concat([combined_arrearage_counts, df])
    combined_arrearage_counts = combined_arrearage_counts.unique()
else:
    combined_arrearage_counts = df

processed_path.unlink(missing_ok=True)
combined_arrearage_counts.write_ipc(processed_path)
combined_arrearage_counts.write_csv(CLEAN_UTILITY_DATA / "arrearage_counts.csv")

# Data Analysis

In [ ]:
arrearage_counts = pl.read_ipc(CLEAN_UTILITY_DATA / "arrearage_counts.arrow")

In [ ]:
arrearage_counts.sum()

Utility,Year,Zip Code,Month,Arrearage Count
str,i64,str,i64,i64
null,48449473,null,147130,15330668


In [ ]:
arrearage_counts.group_by("Month").agg(pl.sum("Arrearage Count")).sort("Month")

Month,Arrearage Count
i8,i64
1,1399354
2,1436629
3,1455351
4,1438461
5,1461971
…,…
8,1140793
9,1153825
10,1101301


In [ ]:
arrearage_counts.group_by("Utility").agg(pl.sum("Arrearage Count")).sort("Utility")

Utility,Arrearage Count
str,i64
"""Avista""",1582131
"""CNG""",751089
"""NWN""",480730
"""PAC""",681597
"""PSE""",11835121


In [ ]:
arrearage_counts.group_by("Utility", "Month").agg(pl.sum("Arrearage Count")).sort(["Utility", "Month"])

Utility,Month,Arrearage Count
str,i8,i64
"""Avista""",1,135957
"""Avista""",2,138516
"""Avista""",3,140037
"""Avista""",4,139427
"""Avista""",5,158139
…,…,…
"""PSE""",8,884871
"""PSE""",9,886455
"""PSE""",10,894789
